# EXP-015 dev-small 확인: dense·BM25 점수 기반 결합

EXP-014(RRF k 20)와 EXP-015(정규화 점수의 가중 합, α 0.5)의 dev-small 지표, 질의별 상위 순위 이동, 목록별 정규화 점수의 모양을 본다. 판정에 쓰지 않고, α·정규화는 여기서 크게 무너질 때만 바꾼다(`docs/experiments/EXP-015.md`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever
from rag.search import query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
PREFIX = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256'
RUN_FILES = {'EXP-014 (RRF k 20)': f'{PREFIX}-k20-dev-small.csv',
             'EXP-015 (점수 결합)': f'{PREFIX}-cc-dev-small.csv'}

## 지표

In [3]:
gold = load_gold(load_cfg(), 'dev-small')
ranks = {name: gold_ranks(read_run(RUNS / f), gold, 10) for name, f in RUN_FILES.items()}
print(len(gold), 'queries')
table = pd.DataFrame({n: {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
                      for n, r in ranks.items()})
print(table.round(4))

1039 queries
           EXP-014 (RRF k 20)  EXP-015 (점수 결합)
recall@1               0.9259           0.9384
recall@5               0.9875           0.9885
recall@10              0.9904           0.9923


## 기준 대비 질의별 이동

In [4]:
b, c = ranks['EXP-014 (RRF k 20)'], ranks['EXP-015 (점수 결합)']
for k in (1, 5):
    gain = int(((b.isna() | (b > k)) & (c <= k)).sum())
    loss = int(((b <= k) & (c.isna() | (c > k))).sum())
    print(f'top{k}: 진입 {gain}, 이탈 {loss}, 순증 {gain - loss}')
print('정답 순위가 바뀐 질의', int((b.fillna(99) != c.fillna(99)).sum()))

top1: 진입 21, 이탈 8, 순증 13
top5: 진입 1, 이탈 0, 순증 1
정답 순위가 바뀐 질의 39


## 목록별 정규화 점수의 모양 (라벨 미사용)

목록마다 2위·5위·10위 문서의 정규화 점수(1위 1, 100위 0). 값이 높으면 상위가 평평하고 낮으면 1위가 두드러진다. 한쪽이 늘 평평하면 다른 쪽이 결합을 지배한다.

In [5]:
cfg = load_cfg(['+exp=exp015', 'index.scope=dev-small'])
ret = Retriever(cfg)
qids, text_path = query_list(cfg)
texts = read_texts(text_path, qids)
emb = query_embeddings(cfg, qids, text_path, query_files(cfg)['emb'])
tokens = ret.query_tokens(texts)
rows = []
for j in range(len(qids)):
    row = ret.chunk_emb @ emb[j]
    d = np.full(len(ret.doc_names), -np.inf, dtype=row.dtype)
    np.maximum.at(d, ret.chunk_doc, row)
    bm = ret.bm25.scores(tokens[j])
    b2 = np.full(len(ret.doc_names), -np.inf, dtype=bm.dtype)
    np.maximum.at(b2, ret.bm25_doc, bm)
    for name, s in [('dense', d), ('bm25', b2)]:
        top = np.sort(s)[::-1][: cfg.retriever.fusion.depth]
        span = top[0] - top[-1]
        norm = (top - top[-1]) / span if span > 0 else np.zeros_like(top)
        rows.append({'list': name, **{f'@{r}': float(norm[r - 1]) for r in (2, 5, 10)}})
shape = pd.DataFrame(rows)
print(shape.groupby('list').median().round(3))

          @2     @5    @10
list                      
bm25   0.469  0.275  0.193
dense  0.572  0.358  0.261
